## Lab 04

## Zadanie 1 (Igor Sieradzki)

Funkcja <t>flow_rate</i> do modyfikacji

In [14]:
def flow_rate(weight, time,*,period = 60,units_per_kg=1000):
    """ Funkcja wylicza ile wagi produktu przybyło/ubyło w jednostce czasu """
    return weight / time * period * units_per_kg

weight = 0.5
time = 3

In [15]:
flow = flow_rate(weight, time)
print("{0:.3} kg per second".format(flow))

1e+04 kg per second


In [ ]:
Zmodyfikuj funckje <i>flow_rate</i> tak, aby poniższe wywołania działały poprawnie.

In [16]:
flow = flow_rate(weight, time, period=60, units_per_kg=1000)
print("{} grams per minute".format(flow))

10000.0 grams per minute


In [17]:
flow = flow_rate(weight, time, period=1, units_per_kg=1)
print("{0:.3} kg per second".format(flow))

0.167 kg per second


In [18]:
flow = flow_rate(weight, time)
print("{0:.3} grams per minute".format(flow))

1e+04 grams per minute


Wyjaśnić czemu poniższe rzucanie błędu jest porządane

In [20]:
try:
    flow = flow_rate(weight, time, 3600, 2.2)
    print("NO ERROR!")
except TypeError:
    print("Żeby nie pomylić jednostek")

Żeby nie pomylić jednostek


## Zadanie 2 (Igor Sieradzki)

Dopisz definicje dekoratora _timeit_, który wypisze na wyjscie standardowe czas wywołania udekorowanej funckji np.

```
@timeit
def foo(x):
    return x**2
    
r = foo(2)
```
wypize: <br>
    `Function foo took: 0.00001 seconds` <br>
oraz zwróci 4

In [21]:
from time import time
from functools import wraps
from time import perf_counter
def timeit(func):
    """ Wypisuje czas wywołania udekorowanej funckji """
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        res =  func(*args, **kwargs)
        elapsed = perf_counter() - start
        print(f"Function {func.__name__} took: {elapsed:.5f} seconds")
        return res
    return wrapper

@timeit
def squares_list(n):
    squares = []
    for i in range(n):
        squares.append(i ** 2)
    return squares

@timeit
def squares_comprehension(n):
    return [i ** 2 for i in range(n)]

@timeit
def squares_map(n):
    return map(lambda x: x**2, range(n))

n = 1000000
l = squares_list(n)
c = squares_comprehension(n)
m = squares_map(n)

Function squares_list took: 0.05167 seconds
Function squares_comprehension took: 0.04201 seconds
Function squares_map took: 0.00000 seconds


## Zadanie 3 (Igor Sieradzki)

Dopisz definicje dekoratora _derivate_ wg. instrukcji w _docstringu_

In [28]:
import sys
sys.float_info.epsilon # epsilon maszynowy


def derivate(epsilon=1000*sys.float_info.epsilon):
    """
    Zwraca pochodną funkcji w punkcie, wg. wzoru f'(x) = [f(x+h) - f(x)]/h, 
    gdzie h jest parametrem dekoratora, jeśli nie zostanie podany, należy przyjąć 1000 * epsilon maszynowy
    """
    h = epsilon
    def decorator(f):
        @wraps(f)
        def wrapper(x):
            return (f(x + h) - f(x))/h
        return wrapper
    return decorator

@derivate(0.01)
def f(x):
    return x*x

@derivate(0.00001)
def g(x):
    return x*x*x+3

def test(a, b, eps=1):
    return abs(round(a)-round(b)) < eps

print(test(f(100), 200.0))
print(round(f(0)) == 0.0)

print(test(g(100), 30000.0))
print(round(g(0)) == 0.0)

from random import random
for x in [random()*1000. for _ in range(20)]:
    print(f(x), 2*x, '\t', test(f(x), 2*x))
    print(g(x), 3*x**2, '\t', test(g(x), 3*x**2))

True
True
True
True
1926.0449321591295 1926.0349321524538 	 True
2782207.942008972 2782207.9199036304 	 True
1444.46119844215 1444.451198444711 	 True
1564829.462766647 1564829.4485162715 	 True
1951.5149575308897 1951.5049575359228 	 True
2856278.7175178523 2856278.699465463 	 True
133.4850938515956 133.4750938514937 	 True
13361.702515976502 13361.70050899879 	 True
961.2762500095414 961.2662500115197 	 True
693024.6159434319 693024.6025584071 	 True
1459.7197333234362 1459.7097333194183 	 True
1598064.3987655637 1598064.3791605853 	 True
1483.732138038613 1483.7221380443593 	 True
1651073.5511779783 1651073.5371921938 	 True
1446.5973576705437 1446.5873576683891 	 True
1569461.25626564 1569461.237524509 	 True
888.0504011642188 888.0404011647611 	 True
591461.8283510208 591461.8155756523 	 True
1928.785101720132 1928.7751017212804 	 True
2790130.066871643 2790130.044764952 	 True
697.0193346045562 697.0093346061701 	 True
364366.5194511413 364366.50939610205 	 True
1749.908016051631

## Zadanie 4 (Igor Sieradzki)

Dopisz definicje dekoratora _accepts_ wg. instrukcji w _docstringu_

In [37]:
def accepts(*types):
    """Sprawdza czy udekorowanej funckji zostały podane odpowiednie parametry zdefiniowane 
       w argumentach dekoratora"""
    import inspect
    def decorator(func):
        sig = inspect.signature(func)
        def throw():
            raise TypeError("ARGUMENTS DONT MATCH!!!!!!!!!!")
        @wraps(func)
        def wrapper(*args, **kwargs):
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()

            for value, t in zip(bound.arguments.values(), types):
                if not isinstance(value, t):
                    raise TypeError("Wrong argument type")

            return func(*args, **kwargs)
        return wrapper
    return decorator

@accepts(str)
def capitalize(word):
    return word[0].upper() + word[1:]

print(capitalize('ola') == 'Ola')

try:
    capitalize(2)
except TypeError:
    print(True)

@accepts(float, int)
def static_pow(base, exp):
    return base ** exp 

print(static_pow(2., 2) == 4.)
print(static_pow(2., exp=2) == 4.)
print(static_pow(base=2., exp=2) == 4.)

try:
    static_pow('x', 10)
except TypeError:
    print(True)
    
try:
    static_pow(2, 2.2)
except TypeError:
    print(True)

True
True
True
True
True
True
True


## Zadanie 5 (Igor Sieradzki)

Dopisz definicje dekoratora _returns_ wg. instrukcji w _docstringu_

In [39]:
def returns(*types):
    """Sprawdza czy udekorowana funkcja zwraca poprawne argumenty, zdefiniowane w parametrach dekoratora"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            res = func(*args, **kwargs)
            if len(types) == 1:
                correct = isinstance(res, types[0])
            else:
                correct = (isinstance(res, tuple)
                           and len(res) == len(types)
                           and all(isinstance(value, expected) for value, expected in zip(res, types)))
            if not correct:
                raise TypeError("AAAAAAAAAAAA")
            return res
        return wrapper
    return decorator


@returns(str)
def str_only_identity(word):
    return word

print(str_only_identity('hello') == 'hello')

try:
    str_only_identity(10)
except TypeError:
    print(True)
    
@returns(int, int)
def split_indices(x):
    return x[0], x[1]

print(split_indices(x=[6,9]) == (6,9))

try:
    split_indices('AB')
except TypeError:
    print(True)

True
True
True
True


## Zadanie 6 (Igor Sieradzki)
Stwórz dekorator cached służący do cachowania wywołań dowolnej funkcji, tzn. chcemy by:
* wywołanie funkcji z określonymi argumentami miało miejsce tylko raz
* funkcja mogła przyjmować dowolną liczbę nazwanych i nienazwanych argumentów
* nie musi reagować poprawnie na domyślne argumenty, tzn. wywołanie funkcji z domyślnymi argumentami a podanie dokładnie takich samych może być traktowane jako dwa różne wywołania
* na opakowanej funkcji można wywołać `.cache_reset()`, który usunie cache z pamięci
* wywołanie `.cache_status()` zwraca string z opisem w postaci: <br>
    `Function FUNCTION_NAME called X times, evaluated Y times`

In [42]:
from random import random
def cached(func):
    cache = {}
    calls = 0
    evaluations = 0
    @wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal calls, evaluations
        calls += 1
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            evaluations += 1
            cache[key] = func(*args, **kwargs)
        return cache[key]

    def cache_reset():
        cache.clear()

    def cache_status():
        return (f"Function {func.__name__} called {calls} times, "
                f"evaluated {evaluations} times")

    wrapper.cache_reset = cache_reset
    wrapper.cache_status = cache_status
    return wrapper
@cached
def foo(x, y=1, z=4):
    return random()
        
print(foo(3) == foo(3))
print(foo(4) == foo(4))
print(foo(3, z=-1, y=3) == foo(3, y=3, z=-1))
print(foo(3) != foo(x=3))
a = foo(3)
foo.cache_reset()
print(a != foo(3))
print(foo.cache_status() == 'Function foo called 10 times, evaluated 5 times')

True
True
True
True
True
True


## Zadanie 7 (Krzysztof Hajto)

Napisz dekorator który będzie robić n-krotne złożenie funkcji, gdzie n jest parametrem dekoratora

In [48]:
def zlozenie(n):
    def decorator(func):
        @wraps(func)
        def wrapper(x):
            res = x
            for i in range(n):
                res = func(res)
            return res
        return wrapper
    return decorator

@zlozenie(3)
def f1(x):
    return x+1

@zlozenie(2)
def f2(x):
    return x*x

@zlozenie(5)
def f3(word):
    return "".join(chr(ord(l)+1) for l in word)

print(f1(2)==5)
print(f2(3)==81)
print(f3("alamakota")=="fqfrfptyf")

True
True
True


## Zadanie 8 (Krzysztof Hajto)

Python nie ma wbudowanej instrukcji switch. Ale posiada anonimowe funkcje oraz słowniki. Zaimplementuj poniższy switch w postaci słownika funkcji.
`
int my_function(x, y) {


    switch(x) {
        case 1: return y*y;
        case 2: return x+y;
        case 3: return x*y;
        case 4: return 0;
    }
}
`

PS. Nigdy nie róbcie tego w faktycznym kodzie :)

In [49]:
def my_function(x, y):
    operations = {
        1: lambda: y * y,
        2: lambda: x + y,
        3: lambda: x * y,
        4: lambda: 0,
    }
    return operations.get(x, lambda: None)()

print(my_function(1,3)==9)
print(my_function(2,4)==6)
print(my_function(3,1)==3)
print(my_function(4,9)==0)

True
True
True
True
